# HistAgent Colab quick start

This notebook runs the released HistAgent model on paired local and context H&E images and generates an ordered molecular readout of up to 50 genes.

[![GitHub](https://img.shields.io/badge/GitHub-HistAgent-181717?logo=github)](https://github.com/zipging/HistAgent) [![Model](https://img.shields.io/badge/%F0%9F%A4%97-Model-FFD21E)](https://huggingface.co/wli13/HistAgent) [![Data](https://img.shields.io/badge/%F0%9F%A4%97-Example_data-FFD21E)](https://huggingface.co/datasets/wli13/HistAgent-data)

> HistAgent is intended for research use. Its generated ranked genes are predictions from histology, not measured transcript counts.

## 1. Prepare the runtime

In Colab, select **Runtime → Change runtime type → GPU** before continuing. HistAgent uses the official gated [Prov-GigaPath](https://huggingface.co/prov-gigapath/prov-gigapath) encoder. Request access on that page before running this notebook.

In [ ]:
%pip install --quiet "git+https://github.com/zipging/HistAgent.git@f93e130" pandas

## 2. Authenticate with Hugging Face

Create a read token whose settings enable **Read access to contents of all public gated repositories you can access**. In Colab, save it under the secret name `HF_TOKEN` using the key icon in the left sidebar. The token is read at runtime and is not written into the notebook.

In [ ]:
import os
from getpass import getpass

from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = getpass("Hugging Face read token: ")

login(token=hf_token, add_to_git_credential=False)
print("Hugging Face authentication configured for this runtime.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A GPU runtime is required. Select Runtime → Change runtime type → GPU.")

device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Load a paired H&E example

HistAgent receives two images centred on the same tissue location: a spot-centred **local view** and a broader **context view**. The example below is a human brain location distributed with the HistAgent tutorial data.

In [ ]:
from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image

data_repo = "wli13/HistAgent-data"
local_path = hf_hub_download(
    data_repo,
    "tutorials/figure5_he_query_brain_local.png",
    repo_type="dataset",
)
context_path = hf_hub_download(
    data_repo,
    "tutorials/figure5_he_query_brain_context.png",
    repo_type="dataset",
)

local_image = Image.open(local_path).convert("RGB")
context_image = Image.open(context_path).convert("RGB")
display(local_image.resize((320, 320)), context_image.resize((320, 320)))

## 4. Load HistAgent

The loader downloads the HistAgent checkpoint from `wli13/HistAgent` and the official GigaPath base encoder. The first run downloads several gigabytes and can take a few minutes.

In [ ]:
from histagent import load_pretrained

torch.set_float32_matmul_precision("high")
model, tokenizer, config = load_pretrained(
    "wli13/HistAgent",
    token=hf_token,
    device=device,
)

print(f"Loaded HistAgent with vocabulary size {config.vocab_size:,}.")

## 5. Generate the ranked molecular readout

Greedy decoding masks special tokens and genes that have already been generated, so the ordered readout does not contain duplicate genes.

In [ ]:
import pandas as pd
from histagent import predict_ranked_genes

genes = predict_ranked_genes(
    model,
    tokenizer,
    local_image,
    context_image,
    organ="brain",
    species="human",
    top_k=50,
    device=device,
)

ranked_readout = pd.DataFrame(
    {"rank": range(1, len(genes) + 1), "gene": genes}
)
display(ranked_readout)
print("Ordered gene sentence:\n", " ".join(genes))

## 6. Save the result

The CSV preserves the gene order and can be used as input to downstream rank-based analyses.

In [ ]:
output_file = "histagent_ranked_molecular_readout.csv"
ranked_readout.to_csv(output_file, index=False)
print("Saved:", output_file)

try:
    from google.colab import files
    files.download(output_file)
except Exception:
    pass

## Use your own images

Replace `local_image` and `context_image` with PIL images or file paths. Both views must be centred on the same tissue location. HistAgent centre-crops each view to 224 × 224 pixels during preprocessing. Use one of the supported species labels (`human`, `mouse` or `unknown`) and a canonical organ label from the released vocabulary.